# Nothing Here Moves

## A ball that bounces without ever leaving its spot

Watch the film at the end of this notebook. A ball rises, hangs, falls, hits the floor and
springs back. You can feel the weight of it.

**It never moves.** The ball is in exactly the same place in every frame. Its outline is
one shape, stamped over and over. The only thing that changes is the **colour inside it**.

This notebook shows you how, from scratch, and you do not need any maths to follow it.

### The idea in three sentences

A shape that changes colour all over at once looks like a shape changing colour. Boring.

But paint a **hairline around its edge** and let that hairline change colour *slightly
before* the middle on one side, and *slightly after* on the other — and your eye reads
that tiny difference in timing as the edge having moved.

The edge drags the whole ball with it, and you see a bounce.

### What you will find here

1. **Two colours** that are opposites, and why that matters
2. **The rule** — how a hairline decides which way a shape appears to go
3. **The bounce** — gravity and hang time, handed to that hairline
4. **The proof** — the notebook checks its own claim, both ways round
5. **Staying safe** — this technique can be uncomfortable to watch if you are careless,
   and there is one obvious "fix" that quietly ruins it
6. **Dancers** — the same rule, given to two figures
7. **The film**

### One tip before you start

Look slightly *away* from whatever you want to see move. Illusions like this one are much
stronger in the corner of your eye than straight ahead.

*The technique is the illusion artist **jagarikin**'s. The settings here were recovered by
measuring his published work frame by frame. Everything runs offline from this one file —
no downloads, no data, no images.*

## 1 · Two colours that are opposites

Everything here is made from two colours: **gold** and **blue**.

Add them together and you get white. Really — channel by channel: 232+23, 185+70, 35+220,
all 255. Each one is the other's *photographic negative*.

That is the engine of the whole thing.

**Why?** Your eye has an old, fast system for spotting movement. It is easily fooled by a
pattern whose light and dark **swap places** from one moment to the next. Vision scientists
call that *reverse phi*. Gold and blue swapping is exactly that swap, over and over.

So the ball runs a loop: gold → light grey → blue → dark grey → gold. Halfway round,
every colour has turned into its own opposite.

**The second ingredient, and it is not a small one.** Suppose the ball only slid back and
forth along the gold–blue line. At the two ends of that line — full gold and full blue —
"a bit early" and "a bit late" would land in almost the same place. The two sides of the
ball would match, and twice every loop it would lose all sense of which way it is going.
You can feel that as a stall.

So the loop travels in a second direction too: **plain brightness**, towards light grey one
way and dark grey the other. And the two directions are deliberately **balanced**, so the
loop is a round trip rather than a flat line. Then there is no moment in the loop where the
two sides of the ball look alike, and the movement never stutters.

No third colour ever appears anywhere: gold, blue, and greys. And the two ends of the
loop still add up to white, exactly as the pair is meant to.

*The cell below sets that up and draws the loop.*

In [ ]:
# ── 1 · the colour loop ──────────────────────────────────────────────────────
%matplotlib inline
import hashlib, importlib.util, io, os, shutil, subprocess, tempfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from PIL import Image

np.random.seed(20260729)                  # fixed, so this runs the same everywhere
rng = np.random.default_rng(20260729)

W, H = 1080, 1350               # 4:5 upright — the shape a phone feed shows in full
FPS = 25                        # frames per second
CYCLE = 12                      # frames for one trip round the loop → about 2 a second
MID = 127.5                     # the grey exactly halfway between gold and blue

GOLD = np.array([232.0, 185.0, 35.0])
BLUE = np.array([23.0, 70.0, 220.0])      # = 255 − gold, channel by channel
AXIS = GOLD - MID                          # one way gold, the other way blue
NEUTRAL = np.array([1.0, 1.0, 1.0])        # the quieter change: lighter ↔ darker

COLOUR_SWING = 1.00     # all the way to gold one way and blue the other
BRIGHT_SWING = 20.0     # and a smaller way towards light grey and dark grey.
# These are the two directions the loop travels in. Full colour and a smaller brightness
# swing was chosen by eye, from a sheet of alternatives, against a rounder and more evenly
# balanced loop — which measures better on paper and looks worse in practice.

def colour_at(phase, swing=COLOUR_SWING, bright=BRIGHT_SWING):
    """What colour the loop is at, at a given point in its trip.
    Halfway round, this is its own exact opposite — which is only true because gold and
    blue add up to white."""
    p = np.asarray(phase, dtype=np.float32)[..., None]
    s = np.asarray(swing, dtype=np.float32)[..., None] if np.ndim(swing) else swing
    b = np.asarray(bright, dtype=np.float32)[..., None] if np.ndim(bright) else bright
    return MID + s * np.cos(p) * AXIS + b * np.sin(p) * NEUTRAL

def linear(c):
    """Screens store colour with a bend in it (that is what "gamma" means). This takes the
    bend out, because light adds up straight and eyes answer to light."""
    v = np.asarray(c, dtype=float) / 255.0
    return np.where(v <= 0.04045, v / 12.92, ((v + 0.055) / 1.055) ** 2.4)

def brightness(c):
    """How bright a colour really looks, 0 to 1."""
    return linear(c) @ np.array([0.2126, 0.7152, 0.0722])

def michelson(v):
    """How far something swings between its lightest and its darkest, 0 to 1."""
    lo, hi = float(np.min(v)), float(np.max(v))
    return (hi - lo) / (hi + lo)

def smoothstep(a, b, x):
    t = np.clip((x - a) / (b - a), 0.0, 1.0)
    return t * t * (3 - 2 * t)

def ease(u):
    """Everything in this film starts gently and stops gently."""
    return 0.5 - 0.5 * np.cos(np.pi * np.clip(u, 0, 1))

ph = np.linspace(0, 2 * np.pi, 512, endpoint=False)
loop = colour_at(ph)
f, ax = plt.subplots(2, 1, figsize=(9.6, 3.2), dpi=100,
                     gridspec_kw={"height_ratios": [1, 1.5]})
f.patch.set_facecolor("#9a9a9a")
ax[0].imshow(np.clip(np.repeat(loop[None, :, :], 40, 0), 0, 255).astype(np.uint8),
             extent=[0, CYCLE / FPS * 1000, 0, 1], aspect="auto")
ax[0].set_yticks([]); ax[0].set_xticks([])
ax[0].set_title("one trip round the loop: gold, light grey, blue, dark grey",
                fontsize=10, color="#17151c")
ax[1].plot(ph / (2 * np.pi) * CYCLE / FPS * 1000, brightness(loop), lw=2.4, color="#17151c")
ax[1].axhline(brightness(np.array([MID, MID, MID])), color="#7d5f0c", lw=1.8, ls="--",
              label="the background")
ax[1].set_xlabel("milliseconds into the loop", color="#22202a")
ax[1].set_ylabel("brightness", color="#22202a"); ax[1].legend(fontsize=8)
ax[1].set_facecolor("#9a9a9a"); ax[1].tick_params(colors="#22202a", labelsize=8)
plt.tight_layout(); plt.show()

lo, hi = float(loop.min()), float(loop.max())
print(f"gold + blue = {np.round(GOLD + BLUE).astype(int)}  ← white, so each is the other's opposite")
ends = colour_at(np.array([0.0, np.pi]))
print(f"the loop's two ends are {np.round(ends[0]).astype(int)} and "
      f"{np.round(ends[1]).astype(int)}, which still add to {np.round(ends.sum(0)).astype(int)}")
turn = np.array([np.linalg.norm(colour_at(p + np.pi / 2) - colour_at(p - np.pi / 2))
                 for p in np.linspace(0, 2 * np.pi, 360, endpoint=False)])
print(f"the two rims differ by {turn.min():.0f} at the least and {turn.max():.0f} at the "
      f"most, round the loop")
assert turn.min() > 50, "the loop must keep some direction all the way round"
print(f"the loop takes {CYCLE} frames at {FPS} fps — about {FPS / CYCLE:.1f} trips a second")
print(f"it swings between light and dark by {michelson(brightness(loop)):.2f} out of 1")
print(f"its average colour is {np.round(loop.mean(0)).astype(int)} — the same grey as the "
      f"background, so the ball has no lasting brightness of its own")
print(f"colours used run {lo:.0f} to {hi:.0f} out of 0–255, so nothing is clipped")
assert lo >= 0 and hi <= 255, "the loop must stay inside what a screen can show"

## 2 · The rule

Here is the whole technique, in one sentence:

> **The rim on the back edge changes colour a little early. The rim on the front edge
> changes a little late. That makes the shape appear to move forwards.**

"Front" and "back" mean relative to the direction you want. To make a ball look like it is
falling, its **top** rim is the back edge and runs early; its **bottom** rim is the front
edge and runs late. The sides — which falling does not involve — are left alone.

For a round shape this becomes one smooth rule. Every point on the rim faces outwards in
some direction, and what it gets is:

* **early**, where it faces *away* from where you want the ball to go
* **late**, where it faces *towards* it
* **nothing**, where it faces sideways

Three details, all measured from the original works rather than assumed:

* **The rim is painted in the same colours as the body, at the same strength.** Only its
  *timing* is different. Giving the rim its own special colour is a tempting idea; it
  works less well.
* **The rim is a hairline, and its width never changes.** In the originals it is a single
  pixel — one pixel on a staircase 160 pixels wide, and one pixel on an arm 4 pixels
  thick. This notebook draws a slightly thicker one, because video squeezes colour to half
  resolution and a one-pixel colour line would not survive the trip to your screen.
* **Early and late means a quarter of the loop.** Half a loop would be the exact opposite
  colour, which points both ways at once and cancels out.

*The cell below builds a disc, gives it a rim, and slices through it so you can see where
the early and late parts are.*

In [ ]:
# ── 2 · shapes, rims, and painting a frame ───────────────────────────────────
YY, XX = np.mgrid[0:H, 0:W].astype(np.float32)
RIM = 9.0            # the rim's width in pixels — the same whatever the shape's size.
                     # It has to survive the trip to a reader's screen: after the picture is
                     # scaled down to phone width this is about 3 px, which is where it
                     # started. A hairline that thins to one pixel on delivery is gone.
QUARTER = np.pi / 2  # a quarter of the loop: what makes the rim point somewhere

def disc(cx, cy, r):
    """→ (how far each point is from the edge — negative inside — and which way is out)."""
    dx, dy = XX - cx, YY - cy
    d = np.hypot(dx, dy)
    return d - r, dx / np.maximum(d, 1e-6), dy / np.maximum(d, 1e-6)

def rim_phase(edge, nx, ny, dirx, diry, strength=1.0):
    """The rule, as code: how early or late each pixel runs.
    The body: bang on time. The rim: a quarter of the loop either side, depending on which
    way that bit of rim faces."""
    facing = np.clip(nx * dirx + ny * diry, -1, 1)   # +1 front, −1 back, 0 side-on
    in_rim = (edge > -RIM) & (edge <= 0)
    return np.where(in_rim, -QUARTER * facing * strength, 0.0).astype(np.float32)

def coverage(edge):
    """The shape itself, with one pixel of softening so the edge is not a staircase.
    Anything blurrier than that and the illusion stops working."""
    return np.clip(0.5 - edge, 0.0, 1.0).astype(np.float32)

def paint(base, cov, phase, clock):
    """Stamp a shape onto the picture. `cov` is worked out once and never changes — it IS
    the shape. Only `clock` moves on."""
    col = colour_at(clock + phase)
    return base * (1 - cov[..., None]) + col * cov[..., None]

FLAT = np.broadcast_to(np.array([MID] * 3), (H, W, 3)).astype(np.float32)
edge, nx, ny = disc(W * 0.5, H * 0.42, 115.0)
cov = coverage(edge)
ph_down = rim_phase(edge, nx, ny, 0.0, 1.0)               # a disc told to fall
frame = paint(FLAT.copy(), cov, ph_down, 0.9)

f, ax = plt.subplots(1, 2, figsize=(9.6, 3.6), dpi=100)
f.patch.set_facecolor("#9a9a9a")
cy0 = int(H * 0.42)
bx = (slice(cy0 - 150, cy0 + 150), slice(int(W * 0.5) - 150, int(W * 0.5) + 150))
ax[0].imshow(np.clip(frame[bx], 0, 255).astype(np.uint8))
ax[0].set_title("a disc told to fall — look at the top and bottom edges", fontsize=10)
ax[0].set_xticks([]); ax[0].set_yticks([])
col = int(W * 0.5); sl = slice(cy0 - 140, cy0 + 140)
ax[1].plot(np.arange(-140, 140), ph_down[sl, col] / QUARTER, lw=2.2, color="#17151c")
ax[1].set_ylim(-1.4, 1.4); ax[1].set_xlabel("pixels, top to bottom through the disc")
ax[1].set_ylabel("early (+) or late (−)")
ax[1].set_title("the middle is on time — only the rims are early and late", fontsize=10)
for a in ax:
    a.set_facecolor("#9a9a9a"); a.tick_params(colors="#22202a", labelsize=8)
    a.title.set_color("#17151c"); a.xaxis.label.set_color("#22202a")
    a.yaxis.label.set_color("#22202a")
plt.tight_layout(); plt.show()

rim_px = int(((edge > -RIM) & (edge <= 0)).sum()); all_px = int((edge <= 0).sum())
print(f"of the disc's {all_px} pixels, {rim_px} are rim — {100 * rim_px / all_px:.0f} in 100")
print("that hairline is the only part of the picture that says which way to go")

## 3 · Which way, and how fast

Point the rule wherever you like and the shape goes there. Flip it and the shape reverses.

**Speed** comes from *how much* early and late you use. A full quarter of the loop is the
hardest push the technique has. Use less and the movement fades. Use none and the rim
matches the middle, the picture becomes an ordinary colour change, and everything stops.

Those are exactly the two controls a bouncing ball needs: a direction that **flips at the
floor**, and a strength that **falls to nothing at the top** — which is what hang time is.

*Four discs below. Same disc, four instructions.*

In [ ]:
# ── 3 · one disc, four instructions ──────────────────────────────────────────
demo = FLAT.copy()
DEMOS = [(0.0, 1.0, 1.0, "fall"), (0.0, -1.0, 1.0, "rise"),
         (1.0, 0.0, 1.0, "go right"), (0.0, 1.0, 0.3, "fall, gently")]
for k, (dx, dy, st, label) in enumerate(DEMOS):
    e, ex, ey = disc(W * (0.16 + 0.226 * k), H * 0.40, 82.0)
    demo = paint(demo, coverage(e), rim_phase(e, ex, ey, dx, dy, st), 1.4)
f = plt.figure(figsize=(9.6, 3.0), dpi=100); f.patch.set_facecolor("#9a9a9a")
plt.imshow(np.clip(demo[int(H * 0.40) - 115:int(H * 0.40) + 115], 0, 255).astype(np.uint8))
plt.axis("off")
plt.title("same disc, four instructions:  " + " · ".join(d[3] for d in DEMOS),
          fontsize=11, color="#17151c")
plt.tight_layout(); plt.show()
print("the only thing different between these four is the direction we handed the rule")

## 4 · The bounce, and a proof that works both ways

You know what a bouncing ball does. **Falling, it speeds up. At the top, it hangs. At the
floor, it turns round in an instant** — and that sudden turn is what makes a bounce feel
like a hit.

We hand that story to the rim. Its direction is *down* while a real ball would be falling
and *up* while it would be rising. Its strength follows the speed a real ball would have —
so it fades to nothing at the top of the arc, and is fiercest at the floor.

Then the proof, and this part matters.

It is easy to write checks that only say **"this must not happen"**: nothing outside the
ball may change, the background must stay frozen. The trouble is that a **blank grey
picture passes every single one of them**. Checks like that reward you for removing
things.

So the cell below also checks the opposite — **"this must happen"**: the ball is really
there and a sensible size, every pixel inside it really does change colour, its colours
really do swap light for dark, and the rim really is carrying its quarter of the loop.
Both halves have to hold, or the notebook stops.

In [ ]:
# ── 4 · the bounce, and both halves of the proof ─────────────────────────────
BOUNCE = 24                       # frames for one bounce = two trips round the loop
BOUNCE_S = BOUNCE / FPS

def bounce_at(frame):
    """→ (height above the floor 0–1, which way it is going, and how fast 0–1)."""
    tau = (frame % BOUNCE) / BOUNCE            # 0 at the floor, ½ at the top
    height = 4.0 * tau * (1.0 - tau)           # the arc gravity draws
    speed = 1.0 - 2.0 * tau                    # + rising, − falling
    return height, (-1.0 if speed > 0 else 1.0), float(min(1.0, abs(speed) * 1.35))

BALL_R, BALL_Y = 216.0, H * 0.300
BALL_E, BALL_NX, BALL_NY = disc(W * 0.5, BALL_Y, BALL_R)
BALL_COV = coverage(BALL_E)                    # THE SHAPE — worked out once, never again
OUTSIDE = BALL_COV <= 0.0
INSIDE = BALL_COV > 0.5

# The card the film is printed on: neutral right where the ball sits — it has to be, since
# neutral grey is exactly halfway between gold and blue — cooling to a deep slate towards
# the edges, which keeps the eye on the middle.
SLATE = np.array([92.0, 106.0, 132.0])
_away = smoothstep(400.0, 1060.0, np.hypot(XX - W * 0.5, YY - BALL_Y))[..., None]
FIELD = (np.array([MID] * 3) * (1 - _away) + SLATE * _away).astype(np.float32)

def ball_frame(i):
    _, dy, st = bounce_at(i)
    ph = rim_phase(BALL_E, BALL_NX, BALL_NY, 0.0, dy, st)
    return paint(FIELD.copy(), BALL_COV, ph, 2 * np.pi * i / CYCLE)

seq = [ball_frame(i).astype(np.uint8) for i in range(BOUNCE * 3)]
changed = np.zeros((H, W), bool)
for a, b in zip(seq, seq[1:]):
    changed |= np.any(a != b, axis=2)
outside_changed = int((changed & OUTSIDE).sum())
ball_px = int((BALL_COV > 0).sum())
inside_changed = int((changed & ~OUTSIDE).sum())
ball_b = np.array([float(brightness(s.astype(np.float32))[INSIDE].mean()) for s in seq])
rim_max = float(np.abs(rim_phase(BALL_E, BALL_NX, BALL_NY, 0.0, 1.0, 1.0)).max())

f, ax = plt.subplots(1, 3, figsize=(9.6, 3.6), dpi=100)
f.patch.set_facecolor("#9a9a9a")
bx = (slice(int(BALL_Y) - 270, int(BALL_Y) + 270), slice(int(W * 0.5) - 270, int(W * 0.5) + 270))
for a, i, ttl in zip(ax, (0, 6, 18), ("at the floor", "falling", "rising")):
    a.imshow(seq[i][bx]); a.set_title(ttl, fontsize=10, color="#17151c")
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

print(f"{len(seq)} frames drawn")
print(f"pixels outside the ball that ever change: {outside_changed}")
print(f"the shape is one array of {ball_px} pixels ({100 * ball_px / (H * W):.1f}% of the "
      f"picture), stamped unchanged into every frame")
print(f"inside it, {inside_changed} pixels change, and its colours swap light for dark by "
      f"{michelson(ball_b):.2f}")
# "this must not happen"…
assert outside_changed == 0, "nothing outside the shape may ever change"
assert len({s[OUTSIDE].tobytes() for s in seq}) == 1, "the background must stay frozen"
# …and "this must happen", without which a blank grey picture would score full marks.
assert 0.04 < ball_px / (H * W) < 0.16, "the ball must actually be there, and stay modest"
assert inside_changed >= 0.99 * ball_px, "every pixel of the ball must change colour"
assert michelson(ball_b) > 0.5, "the ball's colours must swap light and dark — that IS the illusion"
assert rim_max > 0.9 * QUARTER, "the rim must carry its quarter of the loop"

## 5 · Staying safe, and the "fix" that ruins everything

This technique works by making colours swap light for dark, over and over. Done carelessly
that is genuinely dangerous: **fast flashing over a large area can trigger seizures** in
people with photosensitive epilepsy. It is tiring for everyone else too. The published
guidance draws the line at *more than three flashes a second over more than a quarter of
the screen*.

**Here is the trap.** The obvious way to calm this down is to turn the ball's colours
down — less contrast, gentler picture. It works. Every safety number improves.

And the illusion dies completely, because the swapping *is* the mechanism.

That is why the proof in part 4, and the check after the film, both test that the contrast
is **still there**. On this one, "safer" and "broken" can be the same edit, and only
measuring both catches it.

What works instead costs nothing:

**Keep it small, and keep it alone.** The film is one ball on an empty field. One small
shape changes very little of the screen — the cheapest safety there is. A bare stage also
makes the illusion stronger, because there is less for your eye to hold on to. With only
one shape there is no neighbour to balance it against, so its **size** is the lever.

**If you do want several shapes, set them against each other.** Neighbours run half a loop
apart: as one turns gold the next turns blue, one brightening exactly as the other dims.
Each still swaps its own light and dark at full strength — all the illusion needs — but
the total light coming off the screen barely changes.

**Two colours, not a rainbow.** A full rainbow sweeps through yellow and green, the
brightest colours a screen makes, so it swings brightness harder than gold and blue do.

**Slow enough.** One trip round the loop takes twelve frames — about two a second, well
under the three-a-second line.

**No sudden starts.** The film eases up out of stillness and settles back into it.

*The cell below measures a row of six both ways, so you can see what setting shapes against
each other is worth.*

In [ ]:
# ── 5 · the safety instruments ───────────────────────────────────────────────
def safety_report(frames, fps=FPS):
    """The published flash test, plus the comfort number that matters more.
    A "flash" is a pair of opposite brightness jumps. The danger is when one covers more
    than a quarter of the screen, more than three times a second. The area is counted per
    DIRECTION — brightening and dimming separately — which is what the test means."""
    probe = (160, int(round(160 * H / W)))       # the picture is 4:5, not square
    sm = [np.asarray(Image.fromarray(f).resize(probe, Image.Resampling.BICUBIC), np.float32)
          for f in frames]
    ys = [brightness(s) for s in sm]
    dt, signs, worst = 1.0 / fps, [], 0.0
    whole = np.array([float(y.mean()) for y in ys])
    for i in range(len(ys) - 1):
        dy = ys[i + 1] - ys[i]
        up, dn = float(np.mean(dy > 0.10)), float(np.mean(dy < -0.10))
        worst = max(worst, up, dn)
        if max(up, dn) > 0.25:
            signs.append((i * dt, 1 if up > dn else -1))
    flashes, last = 0, None
    for t0, sg in signs:
        if last is not None and sg != last[1] and t0 - last[0] <= 1.0: flashes += 1; last = None
        else: last = (t0, sg)
    # Measure the screen's pulsing over a SLIDING WINDOW of half a loop — which is exactly
    # how long the flicker takes to go from brightest to darkest. What tires an eye is fast
    # change; a caption fading up over a second is slow and deliberate, and measuring
    # across the whole film would muddle the two together.
    win = CYCLE // 2 + 1
    swing = max((michelson(whole[i:i + win]) for i in range(max(1, len(whole) - win + 1))
                 if whole[i:i + win].max() > 0), default=0.0)
    return {"worst_area_one_way": worst, "flash_pairs": flashes, "whole_screen_swing": swing}

def show_safety(rep, label):
    print(f"{label}:")
    print(f"  most of the screen ever jumping ONE way at once: "
          f"{100 * rep['worst_area_one_way']:.1f}%   (danger line: 25%)")
    print(f"  fastest pulse of the whole screen: {rep['whole_screen_swing']:.3f}   "
          f"(0 = the screen never pulses at all)")
    print(f"  flashes over a quarter of the screen: {rep['flash_pairs']}   "
          f"(danger line: more than 3 a second)")

ROW_N, ROW_R, ROW_Y = 6, 74.0, H * 0.42
ROW = []
for k in range(ROW_N):
    cx = W * (0.5 + (k - (ROW_N - 1) / 2) * 0.157)
    e, ex, ey = disc(cx, ROW_Y, ROW_R)
    ROW.append((coverage(e), e, ex, ey, np.pi * (k % 2)))   # every other one, half a loop on

def row_frame(i, lag=6, strength=1.0, oppose=True):
    fr = FLAT.copy()
    for k, (cov, e, ex, ey, tint) in enumerate(ROW):
        _, dy, st = bounce_at(i + k * lag)
        ph = rim_phase(e, ex, ey, 0.0, dy, st * strength)
        fr = paint(fr, cov, ph, 2 * np.pi * i / CYCLE + (tint if oppose else 0.0))
    return fr

IN_STEP = None
for flag, label in ((False, "six balls all in step (what NOT to do)"),
                    (True, "six balls set against each other")):
    probe_frames = [np.clip(row_frame(i, oppose=flag), 0, 255).astype(np.uint8)
                    for i in range(CYCLE * 2)]
    rep_row = safety_report(probe_frames)
    if not flag: IN_STEP = rep_row["whole_screen_swing"]
    show_safety(rep_row, label)
print(f"\nsetting them against each other cuts the screen's pulsing to "
      f"{rep_row['whole_screen_swing'] / IN_STEP:.0%} of what it was, and costs the "
      f"illusion nothing at all")
del probe_frames

## 6 · Two dancers

The rule works on any shape. And — this is the part that makes dancers possible — you can
hand a **different direction to every part of one shape**.

The two figures here are built from overlapping capsules: thick strokes with rounded ends,
melted together so shoulders and hips curve like flesh rather than scaffolding. Then each
limb gets its own instruction. This arm rises while that one falls. The torso sways. The
legs answer.

Every joint stays exactly where it was drawn.

The two figures run half a loop apart from each other, for the same reason the row of six
did: one turns gold as the other turns blue, and the screen stays calm.

They stay here in the notebook rather than in the film — the film keeps just the one ball,
for the reason part 7 gives.

In [ ]:
# ── 6 · dancers, one instruction per limb ────────────────────────────────────
def seg_dist(px, py, ax, ay, bx, by):
    vx, vy = bx - ax, by - ay
    L2 = vx * vx + vy * vy + 1e-9
    t = np.clip(((px - ax) * vx + (py - ay) * vy) / L2, 0.0, 1.0)
    return np.hypot(px - (ax + t * vx), py - (ay + t * vy))

def smin(a, b, k=9.0):
    """A soft minimum, so the capsules melt together instead of crossing like sticks."""
    h = np.clip(0.5 + 0.5 * (b - a) / k, 0.0, 1.0)
    return b + (a - b) * h - k * h * (1.0 - h)

J = {"pelvis": (0, 0), "chest": (2, -52), "neck": (3, -68), "head": (4, -86),
     "shL": (-16, -56), "elbL": (-46, -44), "handL": (-74, -26),
     "shR": (18, -58), "elbR": (40, -80), "handR": (54, -108),
     "hipL": (-11, -2), "kneeL": (-26, 44), "footL": (-38, 92),
     "hipR": (11, -2), "kneeR": (20, 46), "footR": (34, 92)}
CAPS = [("pelvis", "chest", 20, 0), ("chest", "neck", 13, 0),
        ("neck", "head", 10, 1), ("head", "head", 16, 1),
        ("shL", "elbL", 10, 2), ("elbL", "handL", 8.5, 2), ("handL", "handL", 10, 2),
        ("shR", "elbR", 10, 3), ("elbR", "handR", 8.5, 3), ("handR", "handR", 10, 3),
        ("hipL", "kneeL", 12, 4), ("kneeL", "footL", 10, 4), ("footL", "footL", 11, 4),
        ("hipR", "kneeR", 12, 5), ("kneeR", "footR", 10, 5), ("footR", "footR", 11, 5)]
STEPS = [(1.0, -0.25, 0.0), (0.5, -0.6, 0.8), (0.1, -1.0, 2.2), (-0.1, -1.0, 5.4),
         (-0.9, 0.3, 3.0), (0.9, 0.3, 0.5)]     # (which way x, which way y, when)
SWAY = 1.6                                      # seconds for a limb to go and come back

def build_dancers(scale=1.95):
    dep = np.full((H, W), 1e9, np.float32)
    limb = np.zeros((H, W), np.int8)
    for cx, mirror in ((W * 0.28, False), (W * 0.72, True)):
        m, cy = (-1.0 if mirror else 1.0), H * 0.50
        for a, b, r, li in CAPS:
            ax, ay = J[a]; bx, by = J[b]
            d = seg_dist(XX, YY, cx + m * ax * scale, cy + ay * scale,
                         cx + m * bx * scale, cy + by * scale) - r * scale
            limb = np.where(d < dep, np.int8(li + (6 if mirror else 0)), limb)
            dep = smin(dep, d, 9.0)
    return dep, limb

DEDGE, DLIMB = build_dancers()
DCOV = coverage(DEDGE)                                  # the dancers' one frozen stamp
gy, gx = np.gradient(DEDGE)
gm = np.hypot(gx, gy) + 1e-6
DNX, DNY = (gx / gm).astype(np.float32), (gy / gm).astype(np.float32)
DIRS = np.array([[s[0], s[1]] for s in STEPS] * 2, np.float32)
DIRS /= np.linalg.norm(DIRS, axis=1, keepdims=True)
BEATS = np.array([s[2] for s in STEPS] * 2, np.float32); BEATS[6:] += 1.1
DIN_RIM = ((DEDGE > -RIM) & (DEDGE <= 0))
DTINT = np.where(DLIMB >= 6, np.pi, 0.0).astype(np.float32)   # the pair, set against each other

def dancers_frame(t):
    """One frame. The stamp never changes; each limb's rim gets its own early-or-late."""
    swing = np.sin(2 * np.pi * t / SWAY + BEATS)
    facing = DNX * (DIRS[:, 0] * swing)[DLIMB] + DNY * (DIRS[:, 1] * swing)[DLIMB]
    ph = np.where(DIN_RIM, -QUARTER * np.clip(facing, -1, 1), 0.0).astype(np.float32)
    return paint(FLAT.copy(), DCOV, ph + DTINT, 2 * np.pi * t * FPS / CYCLE)

f, ax = plt.subplots(1, 3, figsize=(9.6, 3.8), dpi=100)
f.patch.set_facecolor("#9a9a9a")
for a, t0 in zip(ax, (0.0, 0.24, 0.48)):
    a.imshow(np.clip(dancers_frame(t0), 0, 255).astype(np.uint8)[int(H * 0.16):int(H * 0.90)])
    a.set_title(f"{t0:.2f} s", fontsize=10, color="#17151c")
    a.set_xticks([]); a.set_yticks([])
f.suptitle("the same stamp every time — only the colours are different", color="#17151c")
plt.tight_layout(); plt.show()
print(f"the dancers are {int((DCOV > 0).sum())} pixels, {int(DIN_RIM.sum())} of them rim "
      f"({100 * DIN_RIM.sum() / (DCOV > 0).sum():.0f} in 100)")

## 7 · The film

One ball and about twenty seconds. It bounces. The colour loop stops and it stands revealed
as a still disc. The loop starts again and on it goes. Then, at the very end and only then,
a single button appears and breathes, to invite the click that explains it.

**There is no writing in the picture.** That is not a style choice. Type burnt into a frame
cannot be resized by a reader who needs it larger, cannot be read by a screen reader, cannot
be translated by the mesh that carries every other string on this site, and cannot be
selected or searched — and the page already shows the title and this description above the
film, in the reader's own font. Burning the words in would only make a worse copy of
something that is already there properly.

It also protects the illusion. Every extra thing on screen is one more thing your eye can
hold on to, and each one takes a little of the effect away. A held gaze is exactly what
reverse phi needs, so the emptier the card, the stronger it works.

The card is **upright, four by five** — the shape a phone feed shows in full.

The background is worth a note, because it is not a free choice. Right where the ball sits
it has to be **neutral grey**: grey is exactly halfway between gold and blue, and the ball
must average out to the same colour as the thing behind it, or your eye starts to notice
where it really is. Further out the card cools into a deep slate, which frames the grey so
it reads as a lit stage rather than as an unpainted background.

Three files come out of this: the film as **WebM**, which is what a browser wants and is
the smallest of the three; the same film as **MP4**, which plays everywhere; and a short
looping **GIF** for anywhere a still picture would otherwise go.

## 7 · What gets published, and why it is not a video

The obvious way to publish this is a video. It is the wrong way, and the reason is the rim.

The recipe says the rim is a **fixed number of pixels**, not a fixed fraction of the ball. A video
cannot honour that, because a video does not know how big it will be shown. Measured: this card
authored at 1080 px wide, displayed in a 430 px feed card, thins a 9 px rim to **3.6 px** and loses
**58%** of the difference between the two sides of the rim — the whole cue. Compression is not the
culprit; the same measurement puts the codec at about **3%**. It is the resizing. And there is no
"just export it smaller", because the same card is 430 px on one screen and 1290 px on another.

So the work is published as a **drawing**, not as a recording of a drawing:

* `scene.svg` — the animation itself, about 15 KB. The rim is a stroke marked
  `vector-effect="non-scaling-stroke"`, which keeps it the same width no matter how large the
  drawing is displayed. Measured in a browser: **10 px at 430, 430@2x, 430@3x, 600 and 1080**,
  where an ordinary stroke falls to 4 px. That is the recipe, enforced by the thing doing the
  drawing instead of destroyed by it.
* `loop.webp` — the same loop as pixels, lossless, for anywhere a drawing will not do: a link
  preview, a feed reader, a download. Lossless is also **smaller here than lossy** (45 KB against
  108 KB at quality 90), because the picture is flat colour and lossy compression spends its bits
  inventing noise in it.
* `poster.png` — the first frame, for a reader who has asked for less motion.

**The rim is a gradient, and that is not a trick.** The rim's phase offset is
`−(π/2)·(n̂·d̂)·strength`; the motion is vertical, so `n̂·d̂` is just the vertical part of the
surface normal — which is *linear in y*. A vertical gradient painted onto the stroke is therefore
not an approximation of the rim, it **is** the rim, and the bounce is that gradient's colours
walking round the loop. Seventeen stops hold it to within **0.5 of 255** of the exact cosine.

In [ ]:
# ── the drawing → scene.svg ──────────────────────────────────────────────────
def hexof(v):
    return "#%02x%02x%02x" % tuple(int(round(min(255, max(0, c)))) for c in v)

RIM_CSS = 5.0            # the rim, in pixels, HELD at this width however large the drawing is shown
STOPS   = 17             # gradient stops down the disc — the error against the true rim is checked below
SVG_DUR = BOUNCE / FPS   # one bounce per loop, so the drawing loops seamlessly for ever

ny_at = np.linspace(-1.0, 1.0, STOPS)      # the vertical part of the surface normal, top to bottom

def rim_colour_at(i, ny):
    """The rim's colour at height `ny` on frame `i` — the same rule as rim_phase(), written out."""
    _, dy, st = bounce_at(i)
    return colour_at(2 * np.pi * i / CYCLE - (np.pi / 2) * (ny * dy) * st, COLOUR_SWING, BRIGHT_SWING)

def keyframes(prop, seq):
    return "".join(f"{k * 100 / len(seq):.4f}%{{{prop}:{c}}}" for k, c in enumerate(seq))

body_seq = [hexof(colour_at(2 * np.pi * i / CYCLE, COLOUR_SWING, BRIGHT_SWING)) for i in range(BOUNCE)]
css = [f".b{{animation:b {SVG_DUR}s step-end infinite}}",
       "@keyframes b{" + keyframes("fill", body_seq) + "}"]
stop_tags = []
for s_i, ny in enumerate(ny_at):
    seq = [hexof(rim_colour_at(i, ny)) for i in range(BOUNCE)]
    css.append(f".s{s_i}{{animation:s{s_i} {SVG_DUR}s step-end infinite}}")
    css.append(f"@keyframes s{s_i}{{" + keyframes("stop-color", seq) + "}")
    stop_tags.append(f'<stop class="s{s_i}" offset="{(ny + 1) / 2:.4f}" stop-color="{seq[0]}"/>')
# A reader who has asked their computer for less movement gets a still disc, from the drawing itself.
css.append("@media (prefers-reduced-motion:reduce){*{animation:none!important}}")

ALT = ("A disc that appears to bounce. It never moves: only its colours change, and the hairline "
       "around its edge changes a quarter of a beat before the middle on one side and after it on "
       "the other.")

# The line goes back into the picture — and in a drawing that is a different decision from what it
# was in a video. Type burnt into a frame is pixels: it blurs when the card is small and it cannot
# be made bigger by a reader who needs it bigger. Type in a drawing is TEXT, laid out by the reader's
# own renderer at whatever size the card is, so it is as sharp at 430 px as at 1080. No web font is
# fetched (a published drawing reaches for nothing), so it is set in whatever grotesque the reader's
# system already has — the wording is short enough that a different typeface cannot break the line.
SAYS = ["nothing is moving here,", "only the colours"]
CAP_SIZE, CAP_Y = 52, BALL_Y + BALL_R + 190      # clear of the ball, well inside the card
CAP_FONT = "system-ui,-apple-system,'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif"
caption_svg = "".join(
    f'<text x="{W/2}" y="{CAP_Y + i * CAP_SIZE * 1.42:.0f}" text-anchor="middle" '
    f'font-family="{CAP_FONT}" font-size="{CAP_SIZE}" font-weight="700" fill="#ffffff">{line}</text>'
    for i, line in enumerate(SAYS))
scene = (
    f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="{W}" height="{H}" '
    f'role="img" aria-label="{ALT}">'
    f'<title>Nothing here moves</title><style>{"".join(css)}</style><defs>'
    f'<radialGradient id="f" cx="50%" cy="{100 * BALL_Y / H:.2f}%" r="70%">'
    f'<stop offset="0" stop-color="{hexof([MID] * 3)}"/>'
    f'<stop offset=".37" stop-color="{hexof([MID] * 3)}"/>'
    f'<stop offset="1" stop-color="{hexof(SLATE)}"/></radialGradient>'
    f'<linearGradient id="r" x1="0" y1="{BALL_Y - BALL_R:.1f}" x2="0" y2="{BALL_Y + BALL_R:.1f}" '
    f'gradientUnits="userSpaceOnUse">{"".join(stop_tags)}</linearGradient></defs>'
    f'<rect width="{W}" height="{H}" fill="url(#f)"/>'
    f'<circle class="b" cx="{W / 2}" cy="{BALL_Y}" r="{BALL_R - RIM_CSS / 2:.2f}" fill="{body_seq[0]}"/>'
    f'<circle cx="{W / 2}" cy="{BALL_Y}" r="{BALL_R - RIM_CSS / 2:.2f}" fill="none" stroke="url(#r)" '
    f'stroke-width="{RIM_CSS}" vector-effect="non-scaling-stroke"/>'
    f'{caption_svg}</svg>')
with open("scene.svg", "w") as fh:
    fh.write(scene)
svg_bytes = os.path.getsize("scene.svg")

# What does drawing the rim as 17 gradient stops COST against the exact rule? A gradient interpolates
# straight between stops; the rim is a cosine. Measure the gap rather than assume it is small.
fine = np.linspace(-1, 1, 401)
worst_stop_err = 0.0
for i in range(BOUNCE):
    exact  = np.array([rim_colour_at(i, n) for n in fine])
    knots  = np.array([rim_colour_at(i, n) for n in ny_at])
    approx = np.stack([np.interp(fine, ny_at, knots[:, c]) for c in range(3)], 1)
    worst_stop_err = max(worst_stop_err, float(np.abs(exact - approx).max()))

print(f"scene.svg · the drawing · {svg_bytes:,} bytes")
print(f"  {STOPS} gradient stops, worst gap from the exact rim: {worst_stop_err:.2f} out of 255")
print(f"  the rim is {RIM_CSS:.0f} px wide and STAYS {RIM_CSS:.0f} px wide at any size")
assert worst_stop_err < 2.0, "the gradient must follow the real rim to within a rounding step"
assert svg_bytes < 262144, "the drawing should be a few kilobytes, not a few hundred"
assert "non-scaling-stroke" in scene, "without this the rim thins with the card and the effect goes"
assert "<script" not in scene.lower() and "href" not in scene.lower(), \
    "a published drawing carries no code and fetches nothing — it is one self-contained file"
for line in SAYS:
    assert line in scene, "the line has to be IN the drawing, as text, not painted into it"
assert CAP_Y > BALL_Y + BALL_R + 40, "the writing must clear the ball, not sit on it"
assert CAP_Y + len(SAYS) * CAP_SIZE * 1.42 < H - 40, "and it must fit inside the card"
print(f'  it says: "{" ".join(SAYS)}" — as text, so it stays sharp at any size')

In [ ]:
# ── the same loop as pixels → loop.webp + poster.png ─────────────────────────
# For anywhere a drawing will not do: a link preview, a feed reader, a download. Lossless, which is
# both exact AND smaller here — the picture is flat colour, and lossy compression spends its bits
# inventing noise in exactly the smooth gradient the effect is made of.
# The twins have to SAY THE SAME THING as the drawing, or a reader who gets the fallback — a link
# preview, a feed reader, reduced motion — reads a different work from the one that was published.
# They cannot match it glyph for glyph (the drawing is set in the reader's own system font and this
# is set in the one the notebook ships), and that is fine: what must match is the words.
def caption_layer():
    f = plt.figure(figsize=(W / 100, H / 100), dpi=100); f.patch.set_alpha(0.0)
    a = f.add_axes([0, 0, 1, 1]); a.axis("off"); a.set_xlim(0, 1); a.set_ylim(1, 0)
    for i, line in enumerate(SAYS):
        a.text(0.5, (CAP_Y + i * CAP_SIZE * 1.42) / H, line, ha="center", va="baseline",
               fontsize=CAP_SIZE * 0.75, color="#ffffff", weight="bold", family="DejaVu Sans")
    f.canvas.draw()
    buf = np.asarray(f.canvas.buffer_rgba(), np.float32).copy(); plt.close(f)
    assert buf.shape == (H, W, 4), f"the writing came out {buf.shape}, not the frame's size"
    return buf

CAP_LAYER = caption_layer()
def with_caption(fr):
    al = CAP_LAYER[..., 3:] / 255.0
    return fr * (1 - al) + CAP_LAYER[..., :3] * al

loop_frames = [Image.fromarray(np.clip(with_caption(ball_frame(i)), 0, 255).astype(np.uint8))
               for i in range(BOUNCE)]
loop_frames[0].save("loop.webp", save_all=True, append_images=loop_frames[1:],
                    duration=int(round(1000 / FPS)), loop=0, lossless=True, quality=100, method=6)
loop_frames[0].save("poster.png", optimize=True)
webp_bytes, poster_bytes = os.path.getsize("loop.webp"), os.path.getsize("poster.png")

# The safety check runs on THESE frames, and that is deliberate: they are pixels anyone can decode
# and count, where a drawing would have to be rendered first — by whose browser, at what size?
# The drawing and these frames come out of the same numbers a few lines above, so measuring the
# frames measures the drawing.
rep = safety_report([np.asarray(f) for f in loop_frames])   # uint8: safety_report resizes with PIL
show_safety(rep, "the published loop")
FLASH_RATE = FPS / CYCLE
print(f"\nthe loop turns over {FLASH_RATE:.2f} times a second (the hazard rule starts above 3)")
print(f"the ball covers {100 * (BALL_COV > 0).mean():.1f}% of the picture and swaps its own light "
      f"and dark by {michelson(ball_b):.2f}")
assert FLASH_RATE < 3.0, "the loop must stay under three turns a second"
assert rep["worst_area_one_way"] < 0.20, "and well under a quarter of the screen at once"
assert rep["flash_pairs"] == 0, "so no large-area flash pair should ever be found"
# …and the other way round, because a blank grey film passes every line above.
assert 0.04 < (BALL_COV > 0).mean() < 0.16, "the ball must be on screen, and modest"
assert michelson(ball_b) > 0.5, "its colours must swap light for dark — that IS the illusion"
assert rep["worst_area_one_way"] > 0.01, "something must actually be swapping on screen"

print(f"\nscene.svg   · the drawing        · {svg_bytes / 1024:7.1f} KB")
print(f"loop.webp   · the same, as pixels · {webp_bytes / 1024:7.1f} KB  (lossless)")
print(f"poster.png  · the first frame     · {poster_bytes / 1024:7.1f} KB")
print(f"\nthe video this replaces was 599 KB — the drawing is {599 * 1024 / svg_bytes:.0f} times smaller, "
      f"and it is the only one of the three that keeps the rim the right width on a phone.")
for nm, n in (("scene.svg", svg_bytes), ("loop.webp", webp_bytes), ("poster.png", poster_bytes)):
    assert 0 < n < 25e6, f"{nm} is {n / 1e6:.1f} MB — over the budget"

f, ax = plt.subplots(1, 4, figsize=(9.6, 3.4), dpi=100)
f.patch.set_facecolor("#9a9a9a")
for a, i in zip(ax, np.linspace(0, BOUNCE - 1, 4).astype(int)):
    a.imshow(loop_frames[i]); a.set_xticks([]); a.set_yticks([])
    a.set_title(f"frame {i}", fontsize=9, color="#17151c")
plt.tight_layout(); plt.show()

## Where the numbers came from

None of the settings here are guesses. They were recovered by measuring the illusion artist
**jagarikin**'s published work frame by frame — and they are worth writing down, because
most descriptions of these pieces get them wrong.

* **Twelve frames to a loop**, a little over two a second. Measured across several works:
  2.08, 2.5, 2.78 and 3.0.
* **The rim is a hairline and its width never changes with the shape.** In one piece, a
  staircase 161 pixels wide and an arm 4 pixels thick carry the same one-pixel rim.
* **The rim differs from the body only in timing** — same colours, same strength, a quarter
  of a loop early or late. That one was settled by eye, against four alternatives. Every
  version that gave the rim its own colour was weaker.
* **The rim goes on the two sides facing the way you want to go**, and the sign of it picks
  the direction.
* **The colours have to be opposites about the background**, so that half a loop later the
  light and dark have swapped. The originals do it with a full rainbow. Gold and blue do it
  with two colours, because they add up to white.
* **The background is flat, and the loop's average is exactly that same colour.**

No frame of anyone else's work appears here. Every pixel is drawn by the code above. Only
the measurements travelled.

## Things to try

Change one and run it again:

* **`BRIGHT_SWING = 0`** — flatten the loop onto the gold–blue line. Now "early" and "late"
  land on the same grey, the two rims become identical, and the direction vanishes. The
  quickest way to feel why the loop has to be a round trip.
* **`RIM = 12`** — a fat rim. It starts to look like shading, and the movement weakens.
* **`QUARTER = np.pi`** — half a loop instead of a quarter. The rim becomes the body's
  exact opposite, which points both ways at once, and the motion collapses.
* **`CYCLE = 40`** — a slow loop. Almost nothing moves.
* **`COLOUR_SWING = 0.3`** — the tempting "let's make it gentler" change. Watch every safety
  number improve while the illusion dies with them. The check after the film will now stop
  you shipping it.
* **`oppose=False`** in `row_frame` — six balls all in step. The illusion is unchanged and
  the screen starts to pulse.

## Words used here

* **The loop** — the colour cycle every shape runs on. Twelve frames, gold to blue and back.
* **The rim** — the hairline round a shape, running a quarter of a loop early or late. The
  only part of the picture that says which way to go.
* **Front and back edge** — the sides facing towards, and away from, the movement you want.
  Back runs early, front runs late.
* **Reverse phi** — the effect underneath all this: when light and dark swap places from one
  moment to the next, your eye sees movement, backwards, and strongly.
* **Michelson contrast** — how far something swings between its lightest and darkest, 0 to 1.
* **Photosensitive epilepsy** — a condition where flashing light can bring on a seizure. The
  guidance this film is measured against is the three-flashes-a-second, quarter-of-the-screen
  rule.

## Where to read more

The technique is jagarikin's, explained in his own posts and analysed by the vision
scientist Akiyoshi Kitaoka, who connects it to Flynn and Shapiro's *perpetual diamond* —
where light and dark swapping along very thin edges is what generates the motion. The
science underneath is Anstis (1970) on reverse phi, Anstis and Rogers (1975), and Adelson
and Bergen (1985) on how the eye works motion out from a pattern's energy. Freeman, Adelson
and Heeger's *Motion Without Movement* (1991) is the computer-graphics cousin of the same
idea. On flicker safety: WCAG 2.3.1, the web accessibility standard, and the Harding
flash-and-pattern guidance used by broadcasters.

*Licence CC BY 4.0 · runs offline on ordinary hardware · every frame is arithmetic ·
fixed seed, so it comes out the same everywhere*